# Modeling – Smoking & Drinking Dataset

## Objective
Train and evaluate baseline and classical machine learning models to predict alcohol
consumption (**DRK_YN**). Models are compared using appropriate classification metrics
under a consistent preprocessing workflow.

## Inputs
- Preprocessed train/validation arrays generated in `02_preprocessing.ipynb` and saved in
  `data/processed/`
- Preprocessing pipeline saved in `artifacts/preprocessor.joblib`

## Output
A set of trained baselines, evaluation metrics, and a shortlist of best-performing models.

## Imports and configuration

In [8]:
# Core libraries
import numpy as np
import pandas as pd

# Paths and persistence
from pathlib import Path
import joblib

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42


## Load preprocessed datasets
The modeling notebook consumes the preprocessed arrays generated in `02_preprocessing.ipynb`.
This ensures that all models are trained and evaluated on the same feature representation.

In [9]:
PROCESSED_DIR = Path("../data/processed")
ARTIFACTS_DIR = Path("../artifacts")

required_files = [
    "X_train.npy", "X_val.npy", "y_train.npy", "y_val.npy"
]

for fname in required_files:
    path = PROCESSED_DIR / fname
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Please run 02_preprocessing.ipynb first."
        )

# Load preprocessed arrays (generated in 02_preprocessing.ipynb)

X_train = np.load(PROCESSED_DIR / "X_train.npy", allow_pickle=False)
X_val   = np.load(PROCESSED_DIR / "X_val.npy", allow_pickle=False)
y_train = np.load(PROCESSED_DIR / "y_train.npy", allow_pickle=False)
y_val   = np.load(PROCESSED_DIR / "y_val.npy", allow_pickle=False)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((793076, 24), (198270, 24), (793076,), (198270,))

## Evaluation utilities
A small helper function is used to compute consistent metrics across models.

In [10]:
def evaluate_binary_classifier(
    model_name: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray | None = None
) -> dict:
    """
    Compute standard evaluation metrics for a binary classification model.

    Parameters
    ----------
    model_name : str
        Identifier for the model.
    y_true : np.ndarray
        Ground truth labels.
    y_pred : np.ndarray
        Predicted class labels.
    y_proba : np.ndarray or None, optional
        Predicted probabilities for the positive class.

    Returns
    -------
    dict
        Dictionary containing evaluation metrics.
    """
    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
    else:
        metrics["roc_auc"] = np.nan
    return metrics

## Baselines

We establish simple baselines to define a minimum performance reference.  
Given that the target is nearly balanced, the `most_frequent` strategy is expected to
achieve ~50% accuracy but poor recall/F1 for the positive class. We also include a
`stratified` baseline as a more informative random reference.

In [11]:
from sklearn.dummy import DummyClassifier

baselines = [
    ("Dummy(most_frequent)", DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)),
    ("Dummy(stratified)", DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)),
]

baseline_results = []

for name, model in baselines:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None
    baseline_results.append(evaluate_binary_classifier(name, y_val, y_pred, y_proba))

pd.DataFrame(baseline_results).sort_values(by="f1", ascending=False).reset_index(drop=True)



,model,accuracy,precision,recall,f1,roc_auc
0,Dummy(stratified),0.499526,0.499341,0.500545,0.499942,0.499526
1,Dummy(most_frequent),0.500187,0.000000,0.000000,0.000000,0.500000


The baseline results confirm that the dataset cannot be meaningfully predicted using
trivial strategies. The `most_frequent` baseline fails to identify the positive class,
while the `stratified` baseline achieves approximately random performance across all
metrics. These results establish a clear lower bound that subsequent models must
significantly outperform.

In [16]:
baseline_df = (
    pd.DataFrame(baseline_results)
    .sort_values(by="f1", ascending=False)
    .reset_index(drop=True)
)
baseline_df


,model,accuracy,precision,recall,f1,roc_auc
0,Dummy(stratified),0.499526,0.499341,0.500545,0.499942,0.499526
1,Dummy(most_frequent),0.500187,0.000000,0.000000,0.000000,0.500000


## Classical ML models (first pass)
We train a small set of classical models using reasonable default hyperparameters to
identify promising candidates for further tuning.


### Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    random_state=RANDOM_SEED
)
# Class weights are not adjusted since the target is approximately balanced

lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)
y_proba = lr.predict_proba(X_val)[:, 1]

evaluate_binary_classifier("LogReg", y_val, y_pred, y_proba)


{'model': 'LogReg',
 'accuracy': 0.7272053260705099,
 'precision': 0.7306671313046419,
 'recall': 0.7193787967466548,
 'f1': 0.7249790252459767,
 'roc_auc': 0.8042828870967818}

### Random Forest Classifier

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    min_samples_leaf=2,
    max_features="sqrt"
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
y_proba = rf.predict_proba(X_val)[:, 1]

evaluate_binary_classifier("RandomForest", y_val, y_pred, y_proba)


{'model': 'RandomForest',
 'accuracy': 0.7372017955313461,
 'precision': 0.7324935931052907,
 'recall': 0.747018103291691,
 'f1': 0.7396845539340831,
 'roc_auc': 0.8190107449792458}

Random Forest provides a strong non-linear baseline and is generally robust to skewed
feature distributions. It also offers interpretability via feature importance.

### HistGradientBoosting

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    random_state=RANDOM_SEED,
    max_depth=8,
    learning_rate=0.1
)

hgb.fit(X_train, y_train)

y_pred = hgb.predict(X_val)
y_proba = hgb.predict_proba(X_val)[:, 1]

evaluate_binary_classifier("HistGradientBoosting", y_val, y_pred, y_proba)

{'model': 'HistGradientBoosting',
 'accuracy': 0.7400312704897363,
 'precision': 0.7361687757007489,
 'recall': 0.7479061131405276,
 'f1': 0.7419910299535479,
 'roc_auc': 0.8228239401651922}

Histogram-based gradient boosting offers a strong non-linear model with good scalability
and competitive performance, making it a solid candidate for further tuning.

While gradient boosting frameworks such as XGBoost are commonly used in similar tasks,
the histogram-based gradient boosting implementation from scikit-learn provides a
competitive and dependency-free alternative. XGBoost may be explored as an extension
in future iterations.

## Model comparison

In [17]:
models = {
    "LogReg": lr,
    "RandomForest": rf,
    "HistGradientBoosting": hgb,
}

model_results = []

for name, model in models.items():
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    model_results.append(evaluate_binary_classifier(name, y_val, y_pred, y_proba))

models_df = pd.DataFrame(model_results)


Baseline models are evaluated separately and later merged with classical models for
comparison. This avoids recomputation and keeps the role of trivial baselines explicit.

In [18]:
results_df = (
    pd.concat([baseline_df, models_df], ignore_index=True)
    .sort_values(by="f1", ascending=False)
    .reset_index(drop=True)
)

results_df


,model,accuracy,precision,recall,f1,roc_auc
0,HistGradientBoosting,0.740031,0.736169,0.747906,0.741991,0.822824
1,RandomForest,0.737202,0.732494,0.747018,0.739685,0.819011
2,LogReg,0.727205,0.730667,0.719379,0.724979,0.804283
3,Dummy(stratified),0.499526,0.499341,0.500545,0.499942,0.499526
4,Dummy(most_frequent),0.500187,0.000000,0.000000,0.000000,0.500000


The comparison shows that all classical machine learning models significantly outperform
the baseline strategies, confirming the presence of meaningful predictive signal in the
data.

Logistic Regression provides a strong linear baseline with balanced precision and recall,
while tree-based ensemble methods further improve performance by capturing non-linear
relationships among clinical variables.

Among the evaluated models, HistGradientBoosting achieves the best overall performance,
with the highest F1-score and ROC-AUC. This suggests that histogram-based gradient
boosting is a promising candidate for further tuning and analysis.


## Detailed evaluation (best model)
We inspect confusion matrix and classification report for the best-performing model.

In [ ]:
best_model_name = "HistGradientBoosting"
best_model = hgb

'HistGradientBoosting'

In [20]:
best_model = hgb

y_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val, y_pred)
cm


array([[72610, 26562],
       [24982, 74116]])

In [21]:
print(classification_report(y_val, y_pred, digits=4))


              precision    recall  f1-score   support

           0     0.7440    0.7322    0.7380     99172
           1     0.7362    0.7479    0.7420     99098

    accuracy                         0.7400    198270
   macro avg     0.7401    0.7400    0.7400    198270
weighted avg     0.7401    0.7400    0.7400    198270



The confusion matrix and classification report indicate a well-balanced performance
between both classes. The model correctly identifies a similar number of drinkers and
non-drinkers, with no strong bias toward either false positives or false negatives.

Precision and recall values are closely aligned for both classes (≈0.73–0.75), resulting
in comparable F1-scores. This suggests that the model maintains a stable trade-off between
identifying drinkers and avoiding misclassification of non-drinkers.

Overall accuracy reaches approximately 74%, and both macro and weighted averages confirm
that performance is consistent across classes. These results indicate that the chosen
decision threshold (0.5) provides a reasonable balance for initial evaluation, making the
model suitable for further refinement and analysis.

## Persist best model
The best-performing model from the initial evaluation is saved to enable reproducible
experiments and serve as a baseline for future tuning and analysis.

In [24]:
best_model_name = "HistGradientBoosting"
best_model = hgb


In [25]:
MODEL_DIR = Path("../artifacts")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model_name": best_model_name,
        "model": best_model,
    },
    MODEL_DIR / "best_model_first_pass.joblib"
)


['..\\artifacts\\best_model_first_pass.joblib']

This persisted model represents the best-performing candidate from the initial modeling
iteration. It will be used as a reference point for subsequent hyperparameter tuning,
threshold optimization, and final evaluation.

## Summary and next steps

This notebook evaluated baseline and classical machine learning models using a consistent
preprocessing pipeline and a fixed train/validation split. Models were compared using
accuracy, precision, recall, F1-score, and ROC-AUC to ensure a balanced assessment of
classification performance.

Baseline strategies confirmed that the task cannot be solved trivially. Classical models
significantly outperformed these baselines, indicating the presence of meaningful
predictive signal in the data. Among the evaluated approaches, ensemble-based methods
achieved the best overall results, with HistGradientBoosting emerging as the strongest
candidate in this initial modeling iteration.

### Next steps
- Hyperparameter tuning for the top-performing models (e.g., HistGradientBoosting,
  Random Forest, Logistic Regression).
- Cross-validation to obtain more robust performance estimates.
- Threshold optimization to analyze precision–recall trade-offs under different decision
  criteria.
- Feature importance and error analysis to improve interpretability and guide further
  refinement.
- Optional exploration of advanced boosting frameworks as an extension.